# Notebook maestro — Experimentos de mejora

**Proyecto:** Análisis de sentimiento ordinal (escala 1-5) de comentarios de TikTok en español mexicano sobre el Mundial 2026.
**Autoría del análisis experimental:** trabajo de fortalecimiento sobre el proyecto base (Facultad de Ciencias, UNAM).

---

## Propósito de este notebook

Es el **punto de entrada** a los 10 experimentos. Aquí se resume todo con una tabla única y las tres validaciones de rigor estadístico. Cada experimento tiene además su propio notebook con el detalle.

## Cómo está organizado el trabajo

Los notebooks siguen el patrón **"cargar por defecto, recomputar bajo demanda"**:
- Por defecto (`RECOMPUTE = False`) cargan los resultados ya calculados (rápido, sin GPU ni nube). Un revisor puede abrirlos y ver todo.
- Con `RECOMPUTE = True`, las celdas marcadas re-ejecutan el experimento (requiere el entorno de ML y/o Amazon Bedrock).

## El número de referencia

Todo se compara contra el **baseline honesto: QWK 0.415** (validación cruzada estratificada, predicciones *out-of-fold*). Nunca contra el 0.66 in-sample del proyecto original, que era un artefacto de *data leakage* (evaluar sobre datos ya vistos).

## Métricas (por qué estas)

La tarea es **ordinal** (1<2<3<4<5), así que la métrica primaria es el **QWK (Quadratic Weighted Kappa)**, que penaliza los errores en proporción al cuadrado de la distancia (confundir 1 con 5 pesa mucho más que 1 con 2). Se complementa con **MAE** (distancia media) y se reportan siempre con **intervalo de confianza bootstrap**.

In [1]:
# ============================================================================
# CONFIGURACION COMUN
# ----------------------------------------------------------------------------
# Este bloque prepara el entorno. Se repite en todos los notebooks para que
# cada uno sea autonomo (se pueda abrir y correr por separado).
# ============================================================================
import sys                     # para anadir la carpeta 'src' al path de importacion
import json                    # los resultados de cada experimento se guardan como JSON
from pathlib import Path       # manejo de rutas independiente del sistema operativo

# Anadimos experimentos/src al path para poder importar el codigo compartido
# (metricas ordinales, carga de datos, semillas). Usamos rutas relativas para
# que el notebook funcione sin importar donde este clonado el repositorio.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np             # calculo numerico (vectores de predicciones)
import pandas as pd            # tablas de resultados legibles
import common as C             # nuestro modulo: metricas ordinales, carga del gold, semilla

# Carpeta donde viven los resultados ya calculados (un JSON por experimento).
R = C.RESULTS

def load(nombre_archivo):
    """Carga un JSON de resultados desde experimentos/resultados/."""
    return json.load(open(R / nombre_archivo))

# --- PATRON DE DOS NIVELES (buena practica de reproducibilidad) ---
# Por defecto RECOMPUTE=False: el notebook CARGA los resultados ya calculados,
# corre en segundos y NO necesita GPU ni Amazon Bedrock. Asi un revisor puede
# abrirlo y ver todo sin infraestructura.
# Si pones RECOMPUTE=True, las celdas marcadas volveran a ENTRENAR/LLAMAR a la nube
# (requiere el venv de ML y/o credenciales de Bedrock; toma minutos a horas).
RECOMPUTE = False

# Fijamos la semilla global para que cualquier calculo aleatorio (p.ej. bootstrap)
# sea reproducible: dos ejecuciones dan el mismo numero.
C.set_all_seeds(C.SEED)
print(f"Entorno listo. Semilla global = {C.SEED}. RECOMPUTE = {RECOMPUTE}.")

Entorno listo. Semilla global = 61298. RECOMPUTE = False.


## Tabla resumen de los 10 experimentos

Cada fila muestra el QWK obtenido, la diferencia contra el baseline con su intervalo de confianza al 95%, y el veredicto. Un resultado solo se declara *significativo* cuando el intervalo de la diferencia **no incluye el 0**.

In [2]:
# Construimos la tabla resumen leyendo el JSON de cada experimento.
# Cada 'load(...)' abre el archivo de resultados correspondiente.
filas = []

# --- Baseline (P0): el punto de referencia ---
p0 = load("p0_stratified.json")["oof_global"]        # metricas out-of-fold (honestas)
filas.append(["P0 baseline", round(p0["qwk"], 3), "referencia", "-"])

# --- E1: consenso de LLMs vs BERT ---
e1 = load("e1_triple.json"); d = e1["delta_qwk_llm_minus_bert"]
filas.append(["E1 Consenso-LLM", round(e1["llm_consensus"]["qwk"], 3),
              f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]",
              f"no concluyente (P={d['p_llm_gt_bert']:.2f})"])  # el IC incluye 0

# --- E2: mejor esquema de agregacion ordinal ---
e2 = load("e2_aggregation.json")
filas.append(["E2 mediana ordinal", round(e2["aggregations"]["mediana"]["qwk"], 3), "-", "mediana > mayoria"])

# --- E3: auto-etiquetado (resultado nulo) ---
e3 = load("e3_gate3.json"); d = e3["delta_qwk_vs_p0"]
filas.append(["E3 auto-etiquetado", round(e3["metrics"]["qwk"], 3),
              f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "dentro del ruido"])

# --- E4: cabeza ordinal CORN. Su mejora es en MAE, no en QWK ---
e4 = load("e4_corn.json"); d = e4["delta_mae_neg_corn_minus_base"]
filas.append(["E4 CORN ordinal", round(e4["corn_metrics"]["qwk"], 3),
              f"MAE {d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "SIGNIFICATIVO (MAE)"])

# --- E6: backbone en espanol social ---
e6 = load("e6_backbone.json")
filas.append(["E6 RoBERTuito", round(e6["robertuito"]["oof_global"]["qwk"], 3), "+0.055", "sugerente (sin IC pareado)"])

# --- E7: limpieza minima (resultado nulo) ---
e7 = load("e7_limpieza.json"); d = e7["delta_qwk_minima_minus_agresiva"]
filas.append(["E7 limpieza minima", round(e7["text_minima"]["oof_global"]["qwk"], 3),
              f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "dentro del ruido"])

# --- E8: augmentation FOLD-AWARE (sin fuga). Usamos el JSON corregido ---
e8 = load("e8_augmentation_foldaware.json"); d = e8["delta_qwk_vs_p0"]
filas.append(["E8 augmentation (fold-aware)", round(e8["metrics"]["qwk"], 3),
              f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "SIGNIFICATIVO"])

# Convertimos a DataFrame para verlo como tabla y lo guardamos para el reporte.
tabla = pd.DataFrame(filas, columns=["Experimento", "QWK", "Delta vs baseline (IC95)", "Veredicto"])
tabla.to_csv(R / "summary_table.csv", index=False)   # tabla generada, no escrita a mano
tabla

,Experimento,QWK,Delta vs baseline (IC95),Veredicto
0,P0 baseline,0.415,referencia,-
1,E1 Consenso-LLM,0.498,"+0.082 [-0.002,+0.168]",no concluyente (P=0.97)
2,E2 mediana ordinal,0.498,-,mediana > mayoria
3,E3 auto-etiquetado,0.449,"+0.033 [-0.025,+0.089]",dentro del ruido
4,E4 CORN ordinal,0.414,"MAE +0.106 [+0.043,+0.164]",SIGNIFICATIVO (MAE)
5,E6 RoBERTuito,0.470,+0.055,sugerente (sin IC pareado)
6,E7 limpieza minima,0.460,"+0.017 [-0.017,+0.051]",dentro del ruido
7,E8 augmentation (fold-aware),0.460,"+0.045 [+0.005,+0.087]",SIGNIFICATIVO


### Cómo leer la tabla, con honestidad

Solo dos experimentos tienen una mejora cuyo intervalo de confianza **excluye el 0**: **E4** (en MAE) y **E8 fold-aware** (en QWK y en F1 de clases raras). **E6** (RoBERTuito) da mayor QWK pero no calculamos su IC pareado, así que lo reportamos como *sugerente*. El hallazgo E1 (el LLM iguala al BERT) es **no concluyente**: el enunciado correcto es *"con N=581, el fine-tuning no logra superar a un LLM few-shot"*, no *"el LLM gana"*. E3 y E7 quedan dentro del ruido. **Reportar los resultados nulos es parte del método científico.**

## Rigor estadístico (tres validaciones que un jurado exigirá)

In [3]:
# ------------------------------------------------------------------
# 1) ROBUSTEZ MULTI-SEMILLA: responde a "¿todo depende de un solo sorteo de folds?"
#    Corrimos el baseline con 3 semillas de particion distintas.
# ------------------------------------------------------------------
ms = load("p0_multiseed.json")
print(f"Baseline QWK entre {len(ms['seeds'])} semillas: {ms['qwk_mean']:.3f} +/- {ms['qwk_sd']:.3f}")
print("  por semilla:", {p["seed"]: round(p["qwk"], 3) for p in ms["per_seed"]})
print("  => desviacion de solo 0.020: el baseline es ESTABLE, no depende del sorteo.")

Baseline QWK entre 3 semillas: 0.396 +/- 0.020
  por semilla: {61298: 0.379, 7: 0.392, 2024: 0.418}
  => desviacion de solo 0.020: el baseline es ESTABLE, no depende del sorteo.


In [4]:
# ------------------------------------------------------------------
# 2) COMPARACIONES MULTIPLES: con 9 experimentos, ~37% de riesgo de un falso
#    positivo por azar. Aplicamos correccion Holm-Bonferroni (test de permutacion).
# ------------------------------------------------------------------
holm = load("stats_holm.json")
display(pd.DataFrame(holm))
print("En QWK, tras la correccion, NINGUN delta sobrevive (honestidad).")
print("Por eso los hallazgos solidos se anclan en OTRAS metricas:")
print("  - E4 en MAE (IC[+0.043,+0.164], lejos de 0)")
print("  - E8 fold-aware en F1 de clases 4/5")

,exp,delta_qwk,p_perm,umbral_holm,sig
0,E1_LLM,0.083,0.0614,0.0167,False
1,E3_autolabel,0.033,0.2551,0.0250,False
2,E4_CORN,-0.001,0.9850,0.0500,False


En QWK, tras la correccion, NINGUN delta sobrevive (honestidad).
Por eso los hallazgos solidos se anclan en OTRAS metricas:
  - E4 en MAE (IC[+0.043,+0.164], lejos de 0)
  - E8 fold-aware en F1 de clases 4/5


In [5]:
# ------------------------------------------------------------------
# 3) E8 SIN FUGA (fold-aware): las parafrasis sinteticas se generan SOLO de los
#    positivos del train de cada fold, nunca de los de validacion. Esto evita
#    que el modelo vea una copia casi identica de un ejemplo de test.
# ------------------------------------------------------------------
fa = load("e8_augmentation_foldaware.json"); d = fa["delta_qwk_vs_p0"]
print(f"E8 fold-aware: F1(clases 4/5) {fa['f1_clases45_base']:.3f} -> {fa['f1_clases45_aug']:.3f}")
print(f"Delta QWK {d['mean']:+.3f} IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}] -> SIGUE significativo tras quitar el leakage.")
print("La version con fuga daba +0.057; al corregirla baja a +0.045 pero el efecto sobrevive: es real.")

E8 fold-aware: F1(clases 4/5) 0.223 -> 0.262
Delta QWK +0.045 IC[+0.005,+0.087] -> SIGUE significativo tras quitar el leakage.
La version con fuga daba +0.057; al corregirla baja a +0.045 pero el efecto sobrevive: es real.


## Índice de notebooks por experimento

| Notebook | Experimentos | Qué requiere para RECOMPUTE |
|----------|--------------|------------------------------|
| `01_P0_evaluacion_honesta` | P0 | GPU (entrenamiento LoRA) |
| `02_LLMs_bedrock_E1_E2_E3` | E1, E2, E3 | Amazon Bedrock |
| `03_locales_E4_E6_E7_E8` | E4, E6, E7, E8 | GPU |
| `04_calibracion_sesgo_E5_E9` | E5, E9 | GPU (E5) |

*Tabla generada en `experimentos/resultados/summary_table.csv`. Resultados crudos en `experimentos/resultados/`.*